# SpendDNA - Rahul Sharma's Wallet Year-End Story
**Name:** VIJAYALAKSHMI S

**Batch:** Data Analytics

**Date:** 05/07/2026

This notebook parses 6 months of Rahul Sharma's UPI/bank transactions, cleans up the messy fields, extracts vendor names, categorises spending, finds anomalies, detects spending archetypes, and prints a final report.

Only pandas and numpy are used - no matplotlib/seaborn, no scikit-learn, no regex.

In [1]:
# SpendDNA - Rahul Sharma's 6-month UPI/bank transaction analysis
# Minor Project 2 - The Unlox Academy

# Only pandas + numpy are used, no matplotlib/seaborn/sklearn/regex,


import pandas as pd
import numpy as np

## FEATURE 1: THE TRANSACTION PARSER

In [15]:
# Goal: read the raw csv, fix the messy Date/Amount/Type columns,
# drop duplicate rows, and end up with one clean DataFrame.

# read the raw file as-is (everything is messy strings at this point)

try:
    df = pd.read_csv("rahul_transactions.csv")
except FileNotFoundError:
    df = pd.read_csv("Data set for DADS June.csv")
raw_row_count = len(df)

# the file has 18 exact duplicate rows -> just drop them
df = df.drop_duplicates()
duplicates_dropped = raw_row_count - len(df)


# fixing the Date column (it has 4 different formats mixed together)
# formats seen in the data:
#   2024-04-12     -> ISO format
#   12/04/24       -> DD/MM/YY
#   12-Apr-24      -> DD-Mon-YY
#   12 Apr 2024    -> DD Mon YYYY
#
# NOTE: pd.to_datetime(..., dayfirst=True) alone does NOT work reliably
# here, because it can still swap day/month even on ISO-looking strings.
# So instead we check the shape of each string ourselves (no regex, just
# plain string checks) and parse each format with its own exact pattern.
def parse_date(text):
    text = str(text).strip()

    # ISO format always starts with a 4 digit year, e.g. 2024-04-12
    if "-" in text and text[:4].isdigit():
        return pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")

    # DD/MM/YY uses forward slashes, e.g. 12/04/24
    if "/" in text:
        return pd.to_datetime(text, format="%d/%m/%y", errors="coerce")

    # DD-Mon-YY uses a dash but does NOT start with a year, e.g. 12-Apr-24
    if "-" in text:
        return pd.to_datetime(text, format="%d-%b-%y", errors="coerce")

    # DD Mon YYYY is space separated with 3 parts, e.g. 12 Apr 2024
    if " " in text and len(text.split(" ")) == 3:
        return pd.to_datetime(text, format="%d %b %Y", errors="coerce")

    # anything that doesn't match any known shape becomes NaT (missing)
    return pd.NaT


df["date"] = df["Date"].apply(parse_date)
unparseable_dates = df["date"].isna().sum()


# fixing the Amount column (3 different formats mixed together)
# ₹2462          -> rupee symbol
# Rs. 1,200      -> Rs. prefix + comma
# 1500.00        -> plain number
# strip out the symbols/commas, then convert to a proper number
df["amount"] = (
    df["Amount"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
unparseable_amounts = df["amount"].isna().sum()


# standardising the Type column
# some rows say "DR"/"CR", others say "Debit"/"Credit" -> make it one format
df["type_clean"] = df["Type"].str.lower().replace({"dr": "debit", "cr": "credit"})


#cleaning up Mode (blank values are just missing data)
df["Mode"] = df["Mode"].replace("", np.nan)


# drop any row where the date or amount could not be parsed at all
df = df.dropna(subset=["date", "amount"])

# also pull out month and hour here so every later feature can use them
# (Time column is already "HH:MM", so the first 2 characters give the hour)
df["month"] = df["date"].dt.month   # 1 = Jan, 2 = Feb, ... 6 = Jun
df["hour"] = df["Time"].str[:2].astype(int)

print("=" * 60)
print("FEATURE 1: TRANSACTION PARSER")
print("=" * 60)
print(f"Parsed {len(df)} transactions across 6 months. "
      f"Dropped {duplicates_dropped} duplicates. "
      f"{unparseable_amounts} unparseable amounts, "
      f"{unparseable_dates} unparseable dates.")

FEATURE 1: TRANSACTION PARSER
Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 0 unparseable dates.


## FEATURE 2: VENDOR EXTRACTOR

In [3]:
# Goal: pull a clean vendor name (like "Swiggy") out of a messy
# description (like "UPI-SWIGGY-1234@HDFCBANK" or "BUNDL Tech P L").
# No regex allowed -> just plain "keyword in text" checks.

# Step 1 (done while building this): print df['Description'].unique()
# and manually inspect all the variants, then group them by vendor.

# dictionary: canonical vendor name -> list of keywords that identify it
vendor_keywords = {
    "Swiggy": ["SWIGGY", "BUNDL"],                       # BUNDL Tech = Swiggy's parent company
    "Zomato": ["ZOMATO"],
    "Zepto": ["ZEPTO"],
    "Blinkit": ["BLINKIT", "GROFERS"],                   # Grofers was rebranded to Blinkit
    "Instamart": ["INSTAMART", "KIRANAKART"],
    "BigBasket": ["BIGBASKET", "INNOVATIVE RETAIL"],     # Innovative Retail Concepts = BigBasket's parent co.
    "DMart": ["DMART", "AVENUE SUPERMARTS"],
    "Amazon": ["AMAZON", "AMZN", "FSN E-COMMERCE"],
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA"],
    "Zerodha": ["ZERODHA"],
    "Groww": ["GROWW", "NEXTBILLION"],
    "Uber": ["UBER"],
    "Ola": ["OLA", "ANI TECHNOLOGIES", "ROPPEN"],        # ANI Tech / Roppen = Ola's registered company names
    "Rapido": ["RAPIDO"],
    "BMTC": ["BMTC", "TUMMOC"],
    "Petrol Pump": ["PETROL", "BPCL", "HP PETROL", "INDIAN OIL", "IOC"],
    "Cafe Coffee Day": ["CAFE COFFEE DAY", "CCD", "COFFEE DAY"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRD WAVE", "THIRDWAVE", "TWC INDIA"],
    "Restaurant": ["RESTAURANT", "MEGHANA", "TRUFFLES", "DINEOUT", "EMPIRE RESTAURANT"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Amazon Prime": ["AMZN PRIME", "AMAZON PRIME"],
    "Hotstar": ["HOTSTAR", "STAR INDIA"],
    "BookMyShow": ["BOOKMYSHOW", "BIGTREE", "BMS MOVIE"],
    "Airtel": ["AIRTEL", "BHARTI AIRTEL"],
    "Jio": ["JIO", "RELIANCE JIO"],
    "Vodafone Idea": ["VODAFONE", "VI POSTPAID", "UPI-VI-"],
    "BESCOM": ["BESCOM", "BANGALORE ELEC"],
    "BWSSB": ["BWSSB"],
    "Rent": ["RENT-LANDLORD"],
    "Salary": ["SALARY"],
    # friend-to-friend transfers -> not a vendor, but a special category
    "P2P Transfer": ["AMAN", "ANKIT", "PRIYA", "NEHA", "KARAN", "VIKAS", "SNEHA"],
    # ATM withdrawals -> also not a vendor
    "Cash Withdrawal": ["ATM-WDL", "ATM"],
}


def extract_vendor(description):
    """Match a messy description to a clean vendor name using keywords."""
    text = str(description).upper()
    for vendor_name, keywords in vendor_keywords.items():
        for word in keywords:
            if word in text:
                return vendor_name
    # nothing matched -> flag it so we can inspect and fix the dictionary later
    return "Uncategorised"


df["vendor_clean"] = df["Description"].apply(extract_vendor)

print("\n" + "=" * 60)
print("FEATURE 2: VENDOR EXTRACTOR")
print("=" * 60)
print("Unique canonical vendors found:", df["vendor_clean"].nunique())
print("\nTop 10 vendors by transaction count:")
print(df["vendor_clean"].value_counts().head(10))

# vendor cleanup audit (bonus) - anything still uncategorised needs a new keyword
uncategorised = df[df["vendor_clean"] == "Uncategorised"]["Description"].unique()
print(f"\nDescriptions still Uncategorised ({len(uncategorised)}):", list(uncategorised))


FEATURE 2: VENDOR EXTRACTOR
Unique canonical vendors found: 35

Top 10 vendors by transaction count:
vendor_clean
Swiggy        223
Zomato        121
Ola           101
Amazon         89
Restaurant     73
Uber           71
Zepto          58
Blinkit        55
Flipkart       47
Starbucks      42
Name: count, dtype: int64

Descriptions still Uncategorised (0): []


## FEATURE 3: CATEGORY TAGGER

In [4]:
# Goal: map every cleaned vendor to one of the 12 spending categories
# (plus the 2 special categories: Personal Transfer, Cash Withdrawal).

vendor_to_category = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",
    "BigBasket": "Groceries",
    "DMart": "Groceries",
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",
    "Zerodha": "Investments",
    "Groww": "Investments",
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",
    "Petrol Pump": "Fuel",
    "Cafe Coffee Day": "Cafe",
    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Restaurant": "Restaurants",
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Amazon Prime": "Subscriptions",
    "Hotstar": "Subscriptions",
    "BookMyShow": "Entertainment",
    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vodafone Idea": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Rent": "Utilities",
    "Salary": "Income",
    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",
}

# .map() turns any vendor not in the dictionary into NaN, so we fill
# those with "Uncategorised" instead of leaving them blank
df["category"] = df["vendor_clean"].map(vendor_to_category).fillna("Uncategorised")

print("\n" + "=" * 60)
print("FEATURE 3: CATEGORY TAGGER")
print("=" * 60)
print(df["category"].value_counts())


FEATURE 3: CATEGORY TAGGER
category
Food Delivery        344
Transport            250
E-commerce           172
Quick Commerce       146
Cafe                  99
Restaurants           73
Utilities             49
Groceries             41
Subscriptions         31
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Name: count, dtype: int64


## FEATURE 4: SPENDING OVERVIEW

In [5]:
# Goal: the headline numbers - credits, debits, savings rate,
# top categories, top vendors.

# split the clean data into debit rows (money going out) and credit rows (money coming in)
debit_rows = df[df["type_clean"] == "debit"]
credit_rows = df[df["type_clean"] == "credit"]

# add up all credits and all debits to get the two headline numbers
total_credits = credit_rows["amount"].sum()
total_debits = debit_rows["amount"].sum()
net_change = total_credits - total_debits          # positive = saved money, negative = overspent
savings_rate = (net_change / total_credits) * 100  # what % of income was actually saved

# Personal Transfer and Cash Withdrawal are not "spending" on a vendor,
# so we exclude them when working out category/vendor percentages
spend_rows = debit_rows[~debit_rows["category"].isin(["Personal Transfer", "Cash Withdrawal"])]

# groupby + sum -> total spent in each category, sorted highest first
category_totals = spend_rows.groupby("category")["amount"].sum().sort_values(ascending=False)
category_pct = (category_totals / category_totals.sum()) * 100  # turn totals into %

# same idea but grouped by vendor instead of category
vendor_totals = spend_rows.groupby("vendor_clean")["amount"].sum().sort_values(ascending=False)
vendor_counts = spend_rows.groupby("vendor_clean")["amount"].count()  # how many orders per vendor

print("\n" + "=" * 60)
print("FEATURE 4: SPENDING OVERVIEW")
print("=" * 60)
print(f"Total credits   : Rs. {total_credits:,.0f}")
print(f"Total debits    : Rs. {total_debits:,.0f}")
print(f"Net change      : Rs. {net_change:,.0f}")
print(f"Savings rate    : {savings_rate:.1f}%")
print(f"Transactions    : {len(df)}")
print(f"Unique vendors  : {df['vendor_clean'].nunique()}")

print("\nTop 5 categories by spend:")
for cat in category_totals.head(5).index:
    print(f"  {cat:<18} Rs. {category_totals[cat]:>10,.0f}   ({category_pct[cat]:.1f}%)")

print("\nTop 5 vendors by spend:")
for vendor in vendor_totals.head(5).index:
    print(f"  {vendor:<18} Rs. {vendor_totals[vendor]:>10,.0f}   ({vendor_counts[vendor]} orders)")


FEATURE 4: SPENDING OVERVIEW
Total credits   : Rs. 509,774
Total debits    : Rs. 1,678,901
Net change      : Rs. -1,169,127
Savings rate    : -229.3%
Transactions    : 1310
Unique vendors  : 35

Top 5 categories by spend:
  E-commerce         Rs.    603,877   (37.5%)
  Investments        Rs.    248,160   (15.4%)
  Food Delivery      Rs.    150,839   (9.4%)
  Utilities          Rs.    149,914   (9.3%)
  Restaurants        Rs.    117,737   (7.3%)

Top 5 vendors by spend:
  Amazon             Rs.    332,826   (89 orders)
  Zerodha            Rs.    210,000   (14 orders)
  Flipkart           Rs.    177,510   (47 orders)
  Restaurant         Rs.    117,737   (73 orders)
  Rent               Rs.    108,000   (6 orders)


## FEATURE 5: MONTHLY TREND ANALYSIS

In [6]:
# Goal: build a category x month spend matrix, then find which
# category is growing fastest and which is shrinking fastest.

# pivot table: rows = category, columns = month, values = total spend
# this is basically an Excel-style pivot: one row per category, one
# column per month, and each cell is the total money spent that month
month_pivot = spend_rows.pivot_table(
    values="amount", index="category", columns="month", aggfunc="sum"
).fillna(0)  # fillna(0) so categories with no spend in a month show 0, not blank

# growth % from the first month present to the last month present, per category
# formula: (new value - old value) / old value * 100
first_month = month_pivot.columns.min()   # should be 1 (January)
last_month = month_pivot.columns.max()    # should be 6 (June)
growth_pct = ((month_pivot[last_month] - month_pivot[first_month]) / month_pivot[first_month]) * 100
growth_pct = growth_pct.replace([np.inf, -np.inf], np.nan).dropna()  # drop categories that started at 0 (divide by zero)

print("\n" + "=" * 60)
print("FEATURE 5: MONTHLY TREND ANALYSIS")
print("=" * 60)
month_names = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun"}

print("Monthly trend for the top category (Food Delivery):")
if "Food Delivery" in month_pivot.index:
    for m in month_pivot.columns:
        amt = month_pivot.loc["Food Delivery", m]
        bar = "#" * int(amt / 3000)  # scale bar length for readability
        print(f"  {month_names.get(m, m)} Rs. {amt:>8,.0f} {bar}")

print(f"\nBiggest growth   : {growth_pct.idxmax()} ({growth_pct.max():.1f}% from {month_names[first_month]} to {month_names[last_month]})")
print(f"Biggest decline  : {growth_pct.idxmin()} ({growth_pct.min():.1f}% from {month_names[first_month]} to {month_names[last_month]})")


FEATURE 5: MONTHLY TREND ANALYSIS
Monthly trend for the top category (Food Delivery):
  Jan Rs.   22,633 #######
  Feb Rs.   23,740 #######
  Mar Rs.   24,803 ########
  Apr Rs.   27,756 #########
  May Rs.   25,408 ########
  Jun Rs.   26,499 ########

Biggest growth   : Cafe (57.2% from Jan to Jun)
Biggest decline  : Fuel (-90.5% from Jan to Jun)


## FEATURE 6: TIME-OF-DAY PATTERNS

In [7]:
# Goal: see WHEN Rahul spends - by hour of day, per category.

# category x hour matrix using groupby (24 hour columns, one row per category)
# this counts HOW MANY orders happened in each category, at each hour of the day
hour_pivot = spend_rows.pivot_table(
    values="amount", index="category", columns="hour", aggfunc="count"
).fillna(0)

print("\n" + "=" * 60)
print("FEATURE 6: TIME-OF-DAY PATTERNS")
print("=" * 60)

# % of Food Delivery orders that happen late at night (9 PM to 2 AM)
food_orders = spend_rows[spend_rows["category"] == "Food Delivery"]
late_night_orders = food_orders[(food_orders["hour"] >= 21) | (food_orders["hour"] < 2)]
late_night_pct = len(late_night_orders) / len(food_orders) * 100
print(f"Food Delivery late-night (9 PM - 2 AM) share: {late_night_pct:.1f}%")

# % of Cafe spend that happens in the morning (8 AM to 11 AM)
cafe_orders = spend_rows[spend_rows["category"] == "Cafe"]
morning_cafe = cafe_orders[(cafe_orders["hour"] >= 8) & (cafe_orders["hour"] <= 11)]
if len(cafe_orders) > 0:
    morning_cafe_pct = len(morning_cafe) / len(cafe_orders) * 100
    print(f"Cafe morning-run (8 AM - 11 AM) share      : {morning_cafe_pct:.1f}%")

# simple printable bar chart of Food Delivery orders across the 24 hours
print("\nFood Delivery orders by hour of day:")
for hr in range(24):
    count = hour_pivot.loc["Food Delivery", hr] if hr in hour_pivot.columns else 0
    bar = "#" * int(count)
    print(f"  {hr:02d}:00  {bar} ({int(count)})")


FEATURE 6: TIME-OF-DAY PATTERNS
Food Delivery late-night (9 PM - 2 AM) share: 20.6%
Cafe morning-run (8 AM - 11 AM) share      : 35.4%

Food Delivery orders by hour of day:
  00:00  ## (2)
  01:00  ######## (8)
  02:00  ## (2)
  03:00  #### (4)
  04:00  ####### (7)
  05:00  ####### (7)
  06:00  # (1)
  07:00  ### (3)
  08:00  ######### (9)
  09:00  ############# (13)
  10:00  ####### (7)
  11:00  ##################### (21)
  12:00  ########################### (27)
  13:00  ############### (15)
  14:00  ######### (9)
  15:00  ################# (17)
  16:00  ########## (10)
  17:00  ########### (11)
  18:00  ############################## (30)
  19:00  ###################################### (38)
  20:00  ########################################## (42)
  21:00  ########################### (27)
  22:00  ######################## (24)
  23:00  ########## (10)


## FEATURE 7: ANOMALY DETECTION

In [8]:
# Goal: flag transactions that are unusually large FOR THEIR CATEGORY,
# using a z-score. A z-score above 2 means "top ~2% of that category".

# mean and std computed separately per category (not across the whole
# dataset) - this is the key detail, otherwise a normal big Amazon
# order would wrongly look the same as a huge Swiggy order.
category_mean = spend_rows.groupby("category")["amount"].transform("mean")
category_std = spend_rows.groupby("category")["amount"].transform("std")

spend_rows = spend_rows.copy()  # avoid a pandas "chained assignment" warning
spend_rows["z_score"] = (spend_rows["amount"] - category_mean) / category_std

anomalies = spend_rows[spend_rows["z_score"] > 2].sort_values("z_score", ascending=False)

print("\n" + "=" * 60)
print("FEATURE 7: ANOMALY DETECTION")
print("=" * 60)
print(f"Anomalous transactions found (z-score > 2): {len(anomalies)}")
print("\nTop 5 anomalies:")
for _, row in anomalies.head(5).iterrows():
    date_str = row["date"].strftime("%d %b")
    print(f"  {date_str} - {row['vendor_clean']:<15} Rs. {row['amount']:>8,.0f}   (z={row['z_score']:.1f})")


FEATURE 7: ANOMALY DETECTION
Anomalous transactions found (z-score > 2): 35

Top 5 anomalies:
  26 Jun - Amazon          Rs.   22,008   (z=4.1)
  07 Feb - Amazon          Rs.   21,986   (z=4.1)
  26 Feb - Restaurant      Rs.    8,383   (z=3.9)
  05 Mar - Amazon          Rs.   19,917   (z=3.6)
  22 Jun - Restaurant      Rs.    7,935   (z=3.6)


## FEATURE 8: SPENDING ARCHETYPE DETECTION

In [9]:
# Goal: apply the 8 archetype rules from the project brief.
# Rahul can match more than one archetype at the same time.

# a few numbers we need, computed once so every rule can reuse them
foodie_pct = category_pct.get("Food Delivery", 0) + category_pct.get("Restaurants", 0) + category_pct.get("Cafe", 0)
qcom_pct = category_pct.get("Quick Commerce", 0)
ecom_pct = category_pct.get("E-commerce", 0)
invest_pct = category_pct.get("Investments", 0)
transport_pct = category_pct.get("Transport", 0)
subscription_vendor_count = df[df["category"] == "Subscriptions"]["vendor_clean"].nunique()


def check_foodie():
    """Food Delivery + Restaurants + Cafe combined > 25% of debits."""
    if foodie_pct > 25:
        return f"THE FOODIE ({foodie_pct:.1f}% on food)"
    return None


def check_quick_commerce_junkie():
    """Quick Commerce > 15% of debits."""
    if qcom_pct > 15:
        return f"THE QUICK COMMERCE JUNKIE ({qcom_pct:.1f}% on Q-com)"
    return None


def check_shopaholic():
    """E-commerce > 15% of debits."""
    if ecom_pct > 15:
        return f"THE SHOPAHOLIC ({ecom_pct:.1f}% on e-commerce)"
    return None


def check_investor():
    """Investments > 15% of debits."""
    if invest_pct > 15:
        return f"THE INVESTOR ({invest_pct:.1f}% on SIPs)"
    return None


def check_late_night_snacker():
    """More than 50% of Food Delivery orders happen between 21:00 and 02:00."""
    if late_night_pct > 50:
        return f"THE LATE-NIGHT SNACKER ({late_night_pct:.1f}% food after 9 PM)"
    return None


def check_cab_commuter():
    """Transport > 10% of debits."""
    if transport_pct > 10:
        return f"THE CAB COMMUTER ({transport_pct:.1f}% on transport)"
    return None


def check_subscription_lover():
    """5 or more distinct subscription vendors active."""
    if subscription_vendor_count >= 5:
        return f"THE SUBSCRIPTION LOVER ({subscription_vendor_count} active subs)"
    return None


def check_yolo_spender():
    """Savings rate below 10%."""
    if savings_rate < 10:
        return f"THE YOLO SPENDER (savings rate {savings_rate:.1f}%)"
    return None


def check_disciplined_saver():
    """Savings rate above 40%."""
    if savings_rate > 40:
        return f"THE DISCIPLINED SAVER (savings rate {savings_rate:.1f}%)"
    return None


# bonus: a 9th, self-invented archetype specific to Bengaluru tech life
def check_brewery_regular():
    """Custom archetype: Cafe spend across 3+ different cafe vendors
    signals someone who treats cafes as a co-working space, very
    common among Bengaluru tech workers."""
    cafe_vendor_count = df[df["category"] == "Cafe"]["vendor_clean"].nunique()
    if cafe_vendor_count >= 3:
        return f"THE PAVEMENT COFFEE CONNOISSEUR ({cafe_vendor_count} different cafes visited)"
    return None


archetype_checks = [
    check_foodie,
    check_quick_commerce_junkie,
    check_shopaholic,
    check_investor,
    check_late_night_snacker,
    check_cab_commuter,
    check_subscription_lover,
    check_yolo_spender,
    check_disciplined_saver,
    check_brewery_regular,
]

print("\n" + "=" * 60)
print("FEATURE 8: SPENDING ARCHETYPE DETECTION")
print("=" * 60)
print("RAHUL'S SPENDING ARCHETYPES:")
matched_any = False
for check in archetype_checks:
    result = check()
    if result:
        print(f"  -> {result}")
        matched_any = True
if not matched_any:
    print("  No archetype matched.")


FEATURE 8: SPENDING ARCHETYPE DETECTION
RAHUL'S SPENDING ARCHETYPES:
  -> THE SHOPAHOLIC (37.5% on e-commerce)
  -> THE INVESTOR (15.4% on SIPs)
  -> THE YOLO SPENDER (savings rate -229.3%)
  -> THE PAVEMENT COFFEE CONNOISSEUR (3 different cafes visited)


## FINAL PRINTED REPORT (pulls everything above into one clean report)

In [14]:
print("\n")
print("=" * 60)
print(f"  SpendDNA REPORT - RAHUL SHARMA")
print(f"  6 months - {len(df)} transactions - Jan to Jun 2024")
print("=" * 60)
print("\n  EXECUTIVE SUMMARY")
print(f"    Total credits           : Rs. {total_credits:,.0f}")
print(f"    Total debits            : Rs. {total_debits:,.0f}")
print(f"    Net change              : Rs. {net_change:,.0f}")
print(f"    Savings rate            : {savings_rate:.1f}%")
print(f"    Transactions            : {len(df)}")
print(f"    Unique vendors          : {df['vendor_clean'].nunique()}")
print("\n  TOP CATEGORIES (% of debit total)")
for cat in category_totals.head(5).index:
    bar = "#" * int(category_pct[cat])
    print(f"    {cat:<16}{bar:<20} {category_pct[cat]:>5.1f}%   Rs. {category_totals[cat]:>10,.0f}")
print("\n  TOP VENDORS")
for vendor in vendor_totals.head(5).index:
    print(f"    {vendor:<16} Rs. {vendor_totals[vendor]:>10,.0f}   ({vendor_counts[vendor]} orders)")
# --- TIME-OF-DAY PATTERNS section (uses the numbers we found in Feature 6) ---
print("\n  TIME-OF-DAY PATTERNS")
print(f"    Food Delivery late-night (9 PM-2 AM) : {late_night_pct:.1f}% of orders")
if len(cafe_orders) > 0:
    print(f"    Cafe morning-run (8 AM-11 AM)         : {morning_cafe_pct:.1f}% of orders")
print(f"    Quick Commerce                        : spread across the day")
# --- MONTHLY TREND section for Food Delivery (uses the pivot table from Feature 5) ---
print("\n  MONTHLY TREND (Food Delivery)")
if "Food Delivery" in month_pivot.index:
    for m in month_pivot.columns:
        amt = month_pivot.loc["Food Delivery", m]
        bar = "#" * int(amt / 3000)  # shrink the number so the bar fits on screen
        print(f"    {month_names.get(m, m)} Rs. {amt:>8,.0f} {bar}")
# --- TOP ANOMALIES section (uses the z-score table from Feature 7) ---
print("\n  TOP ANOMALIES (3+ stddev from category mean)")
top_anomalies = anomalies[anomalies["z_score"] > 3].head(5)  # z > 3 = 3+ std deviations away
if len(top_anomalies) == 0:
    top_anomalies = anomalies.head(5)  # fall back to top 5 anomalies if none are above z=3
for _, row in top_anomalies.iterrows():
    date_str = row["date"].strftime("%d %b")
    print(f"    {date_str} - {row['vendor_clean']:<15} Rs. {row['amount']:>8,.0f}   (z={row['z_score']:.1f})")
print("\n  RAHUL'S SPENDING ARCHETYPES")
for check in archetype_checks:
    result = check()
    if result:
        print(f"    -> {result}")
print("\n" + "=" * 60)
print("  KEY INSIGHTS")
print(f"1. {category_totals.index[0]} is Rahul's biggest spending category at "
      f"{category_pct.iloc[0]:.1f}% of all discretionary spend.")
print(f"2. His savings rate is {savings_rate:.1f}%, meaning he "
      f"{'is saving comfortably' if savings_rate > 0 else 'is spending more than he earns'}.")
print(f"3. {len(anomalies)} transactions were flagged as unusually large "
      f"for their category - worth a closer look.")
print("=" * 60)



  SpendDNA REPORT - RAHUL SHARMA
  6 months - 1310 transactions - Jan to Jun 2024

  EXECUTIVE SUMMARY
    Total credits           : Rs. 509,774
    Total debits            : Rs. 1,678,901
    Net change              : Rs. -1,169,127
    Savings rate            : -229.3%
    Transactions            : 1310
    Unique vendors          : 35

  TOP CATEGORIES (% of debit total)
    E-commerce      #####################################  37.5%   Rs.    603,877
    Investments     ###############       15.4%   Rs.    248,160
    Food Delivery   #########              9.4%   Rs.    150,839
    Utilities       #########              9.3%   Rs.    149,914
    Restaurants     #######                7.3%   Rs.    117,737

  TOP VENDORS
    Amazon           Rs.    332,826   (89 orders)
    Zerodha          Rs.    210,000   (14 orders)
    Flipkart         Rs.    177,510   (47 orders)
    Restaurant       Rs.    117,737   (73 orders)
    Rent             Rs.    108,000   (6 orders)

  TIME-OF-DAY 